# AndinaLog 03B | Inventory_Tracking | Diagnóstico v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-INVENTORY-diagnostico-didactico-v2"
COLUMNAS_BRONZE = ["movimiento_id", "lote_id", "producto_id", "centro_distribucion",
    "fecha_ingreso", "fecha_salida", "fecha_vencimiento", "cantidad_ingreso",
    "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"]
CENTROS = {"Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"}

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz=Path(RUTA_PROYECTO_DRIVE)
        if not (raiz / "datasets/AndinaLog_03B_Bronce/andinalog_inventory_tracking.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets/AndinaLog_03B_Bronce/andinalog_inventory_tracking.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ=encontrar_raiz()
RUTA_BRONZE=RAIZ / "datasets/AndinaLog_03B_Bronce/andinalog_inventory_tracking.csv"
RUTA_PRODUCTOS=RAIZ / "proyecto-integrador/02_tratamiento/andinalog_productos/salidas/andinalog_productos_didactico_v2_silver.csv"
SALIDAS=RAIZ / "proyecto-integrador/01_diagnostico/andinalog_inventory_tracking/salidas"
HASH_BRONZE=hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
bronze=pd.read_csv(RUTA_BRONZE,dtype="string",encoding="utf-8-sig",keep_default_na=False)
if list(bronze.columns)!=COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df=bronze.copy(deep=True)
df.insert(0,"fila_bronze",range(1,len(df)+1))
print("Filas Bronze:",len(df))


## Reglas de completitud, formato y unicidad

Las comprobaciones leen copias auxiliares; no modifican el Bronze. La primera copia exacta se conserva y se marca crítica solo cada copia posterior. Si una clave se repite con atributos distintos, se marcan todas sus variantes.


In [ ]:
PRIORIDAD={"OK":0,"NO_EVALUABLE":1,"REVISAR":2,"CRITICO":3}
for c in COLUMNAS_BRONZE:
    df[f"{c}_estado"]="OK"
    df[f"{c}_motivo"]=""

def marcar(columna,mascara,estado,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    e,m=f"{columna}_estado",f"{columna}_motivo"
    subir=mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir,e]=estado
    previo=df.loc[mascara,m]
    df.loc[mascara,m]=previo.where(previo.eq(""),previo+"; ")+motivo

def texto(c): return df[c].str.strip()

obligatorias=[c for c in COLUMNAS_BRONZE if c!="fecha_salida"]
for c in obligatorias:
    marcar(c,texto(c).eq(""),"CRITICO","Valor faltante")

for c,patron in {"movimiento_id":r"MOV-\d{6}",
                 "lote_id":r"LOT-\d{4}-\d{5}",
                 "producto_id":r"PROD-\d{3}"}.items():
    marcar(c,texto(c).ne("") & ~texto(c).str.fullmatch(patron).fillna(False),
           "CRITICO",f"Formato esperado: {patron}")

marcar("centro_distribucion",texto("centro_distribucion").ne("") &
       ~texto("centro_distribucion").isin(CENTROS),"CRITICO","Centro fuera de los cinco permitidos")

firma=pd.util.hash_pandas_object(df[COLUMNAS_BRONZE],index=False)
copia=df.duplicated(COLUMNAS_BRONZE,keep="first")
for c in ["movimiento_id","lote_id"]:
    clave=texto(c)
    variantes=firma.groupby(clave,dropna=False).transform("nunique")
    conflicto=clave.ne("") & clave.duplicated(keep=False) & variantes.gt(1)
    marcar(c,conflicto,"CRITICO","Clave repetida con datos contradictorios")
    marcar(c,copia,"CRITICO","Copia exacta posterior de una fila Bronze")


## Fechas y secuencia temporal

El formato canónico es `AAAA-MM-DD`. `DD/MM/AAAA` válido se marca para normalización futura. Una fecha imposible es crítica. La fecha de salida vacía indica lote aún en almacén. La salida posterior al vencimiento se informa como riesgo operativo y se conserva para el análisis.


In [ ]:
fechas={}
for c in ["fecha_ingreso","fecha_salida","fecha_vencimiento"]:
    valor=texto(c)
    iso=valor.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
    alterno=valor.str.fullmatch(r"\d{2}/\d{2}/\d{4}").fillna(False)
    fecha_iso=pd.to_datetime(valor.where(iso),format="%Y-%m-%d",errors="coerce")
    fecha_alt=pd.to_datetime(valor.where(alterno),format="%d/%m/%Y",errors="coerce")
    fechas[c]=fecha_iso.fillna(fecha_alt)
    marcar(c,valor.ne("") & alterno & fechas[c].notna(),"REVISAR","Fecha válida con formato DD/MM/AAAA")
    marcar(c,valor.ne("") & fechas[c].isna(),"CRITICO","Fecha imposible o formato no interpretable")

ingreso,salida,vencimiento=[fechas[c] for c in ["fecha_ingreso","fecha_salida","fecha_vencimiento"]]
marcar("fecha_salida",texto("fecha_salida").eq(""),"NO_EVALUABLE","Sin salida registrada; lote posiblemente aún en almacén")
marcar("fecha_salida",ingreso.notna() & salida.notna() & salida.lt(ingreso),"CRITICO","Salida anterior al ingreso")
marcar("fecha_vencimiento",ingreso.notna() & vencimiento.notna() & vencimiento.lt(ingreso),
       "CRITICO","Vencimiento anterior al ingreso")
marcar("fecha_salida",salida.notna() & vencimiento.notna() & salida.gt(vencimiento),
       "REVISAR","Salida posterior al vencimiento (riesgo de negocio)")


## Cantidades, costo y coherencia del lote

Las cantidades son unidades de inventario del producto. No se presupone que sean kg, cajas o piezas. `cantidad_ingreso` debe ser positiva; salida y merma pueden ser cero. Un remanente positivo es admisible. Solo salida más merma **mayor** que ingreso indica un saldo incoherente.


In [ ]:
numeros={}
enteros=["cantidad_ingreso","cantidad_salida","cantidad_merma","dias_en_almacen"]
for c in [*enteros,"costo_unitario_bob"]:
    valor=texto(c)
    n=pd.to_numeric(valor,errors="coerce")
    numeros[c]=n
    marcar(c,valor.ne("") & n.isna(),"CRITICO","Valor no numérico")
    if c in enteros:
        marcar(c,n.notna() & n.mod(1).ne(0),"CRITICO","Se requiere un número entero")
        marcar(c,n.lt(0),"CRITICO","Valor negativo no permitido")
    if c in ["cantidad_ingreso","costo_unitario_bob"]:
        marcar(c,n.eq(0),"CRITICO","Valor debe ser mayor que cero")

cant_ing,cant_sal,merma=[numeros[c] for c in ["cantidad_ingreso","cantidad_salida","cantidad_merma"]]
comparable=(cant_ing.notna() & cant_sal.notna() & merma.notna() &
            cant_ing.ge(0) & cant_sal.ge(0) & merma.ge(0))
exceso=comparable & (cant_sal+merma).gt(cant_ing)
marcar("cantidad_salida",exceso,"CRITICO","Salida más merma supera ingreso del lote")
marcar("cantidad_merma",exceso,"CRITICO","Salida más merma supera ingreso del lote")

dias=numeros["dias_en_almacen"]
diferencia=(salida-ingreso).dt.days
marcar("dias_en_almacen",ingreso.notna() & salida.notna() & dias.notna() & dias.ne(diferencia),
       "REVISAR","Días declarados no coinciden con ingreso y salida")
marcar("dias_en_almacen",salida.isna() & texto("fecha_salida").eq(""),
       "NO_EVALUABLE","Sin fecha de salida para contrastar días en almacén")


## Referencia parcial a Productos Silver

La ausencia de un producto en el Silver disponible se marca para revisión. Cuando hay correspondencia, una diferencia entre el costo del movimiento y el costo maestro se marca `REVISAR`: el costo histórico de un lote puede diferir del costo actual del producto, así que no se corrige aquí.


In [ ]:
productos=None
if RUTA_PRODUCTOS.is_file():
    productos=pd.read_csv(RUTA_PRODUCTOS,dtype="string",encoding="utf-8-sig",keep_default_na=False)
    if productos["producto_id"].duplicated().any():
        raise ValueError("Productos Silver tiene producto_id duplicado")
    p=productos.set_index("producto_id")
    clave_producto=texto("producto_id").str.upper()
    formato_producto=texto("producto_id").str.fullmatch(r"PROD-\d{3}").fillna(False)
    sin_correspondencia=formato_producto & ~clave_producto.isin(p.index)
    marcar("producto_id",sin_correspondencia,"REVISAR","Sin correspondencia en Productos Silver de cobertura parcial")
    costo_maestro=pd.to_numeric(clave_producto.map(p["costo_unitario_bob"]),errors="coerce")
    diferencia_costo=(costo_maestro.notna() & numeros["costo_unitario_bob"].notna() &
        numeros["costo_unitario_bob"].sub(costo_maestro).abs().gt(0.01))
    marcar("costo_unitario_bob",diferencia_costo,"REVISAR","Costo difiere del maestro de producto; verificar vigencia")
else:
    print("Productos Silver no disponible: integridad referencial y costo no evaluados")


## Resumen y exportación

La cuarentena de diagnóstico se basa en problemas críticos de calidad. Las salidas posteriores al vencimiento quedan visibles en el diagnosticado como riesgo; el tratamiento decidirá cómo preparar cada dato sin borrar el hecho observado.


In [ ]:
# Public diagnostic contract: one quarantine flag and one reason per Bronze field.
for campo in COLUMNAS_BRONZE:
    df[f"{campo}_en_cuarentena"] = df[f"{campo}_estado"].eq("CRITICO")
df["en_cuarentena"] = df[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1)
columnas_publicas = ["fila_bronze", *COLUMNAS_BRONZE]
columnas_publicas += [x for c in COLUMNAS_BRONZE for x in (f"{c}_en_cuarentena", f"{c}_motivo")]
columnas_publicas += ["en_cuarentena"]
diagnosticado = df[columnas_publicas].copy()
cuarentena = diagnosticado.loc[diagnosticado["en_cuarentena"]].copy()
conteos = {"filas_bronze":len(bronze), "filas_diagnosticadas":len(diagnosticado),
           "filas_cuarentena":len(cuarentena), "filas_con_observaciones":int(df[[f"{c}_motivo" for c in COLUMNAS_BRONZE]].ne("").any(axis=1).sum())}
for campo in COLUMNAS_BRONZE:
    conteos[f"cuarentena_{campo}"] = int(diagnosticado[f"{campo}_en_cuarentena"].sum())
reporte_calidad = pd.DataFrame([{"metrica":k,"valor":v} for k,v in conteos.items()])
assert len(diagnosticado)==len(bronze)
assert diagnosticado["en_cuarentena"].equals(diagnosticado[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1))
assert len(cuarentena)==int(diagnosticado["en_cuarentena"].sum())
pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE],bronze)
SALIDAS.mkdir(parents=True,exist_ok=True)
base="andinalog_inventory_tracking_didactico_v2_"
diagnosticado.to_csv(SALIDAS/(base+"diagnosticado.csv"),index=False,encoding="utf-8-sig")
cuarentena.to_csv(SALIDAS/(base+"cuarentena.csv"),index=False,encoding="utf-8-sig")
reporte_calidad.to_csv(SALIDAS/(base+"reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(conteos)
display(diagnosticado.tail(5))
